# Team AEC Possession Shapes

Find a team's highest `aec_per_throw` scoring possessions and compare them with the middle of the filtered efficient-possession set.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.selection

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.selection)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    build_aec_possession_sets,
    build_scoring_possessions,
    compare_top_aec_metrics_by_team,
    create_team_aec_comparison_browser,
    fetch_shownspace_season_throws_cached,
    plot_possession_path,
    plot_representative_paths,
    select_top_aec_possessions_by_team,
    select_top_aec_possessions_league,
)

In [2]:
TEAM_ID = 'breeze'
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'
METRIC = 'aec_per_throw'
TOP_N = 5
MIDDLE_COUNT = 5

## Load Shown Space Throws

In [3]:
games, throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f'Games loaded: {len(games):,}')
print(f'Throws loaded: {len(throws):,}')
print(f'Scoring possessions found: {len(possessions):,}')

Games loaded: 11
Throws loaded: 6,177
Scoring possessions found: 245


## Build Highest and Middle AEC Sets

Defaults: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [4]:
sets = build_aec_possession_sets(
    possessions,
    paths,
    top_n=TOP_N,
    middle_count=MIDDLE_COUNT,
    metric=METRIC,
)
filtered_possessions, filtered_paths = sets['filtered']
highest_possessions, highest_paths = sets['highest']
middle_possessions, middle_paths = sets['middle']

print(f'Filtered analysis possessions: {len(filtered_possessions):,}')

Filtered analysis possessions: 85


In [5]:
summary_columns = [
    'possession_id', 'GameID', 'line_type', 'start_y', 'end_y',
    'field_progress', 'throw_count', 'huck_count', 'total_aec', METRIC,
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]
highest_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-07-11-PHI-DC|1|2|1|True,2026-07-11-PHI-DC,o_line,20.00,107.71,87.71,4,0,1.013382,0.253345,36.93,0.25,0.250
1,2026-07-03-DC-PHI|2|8|1|False,2026-07-03-DC-PHI,o_line,40.26,103.98,63.72,4,0,1.006642,0.251660,22.52,0.25,0.500
2,2026-06-13-BOS-DC|1|9|1|True,2026-06-13-BOS-DC,o_line,24.78,102.35,77.57,4,0,0.967533,0.241883,38.37,0.50,0.375
3,2026-05-23-NY-DC|4|8|3|True,2026-05-23-NY-DC,o_line,22.51,108.67,86.16,5,0,1.115203,0.223041,35.01,0.30,0.200
4,2026-04-25-DC-BOS|2|9|1|False,2026-04-25-DC-BOS,o_line,21.21,107.78,86.57,5,0,1.000646,0.200129,18.60,0.50,0.000


In [6]:
middle_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-05-10-DC-NY|5|2|1|False,2026-05-10-DC-NY,o_line,2.62,106.54,103.92,11,0,1.012093,0.092008,38.47,0.272727,0.454545
1,2026-05-10-DC-NY|2|11|1|False,2026-05-10-DC-NY,o_line,10.91,106.38,95.47,11,0,1.015889,0.092354,35.19,0.500000,0.272727
2,2026-04-25-DC-BOS|4|4|1|False,2026-04-25-DC-BOS,o_line,17.02,108.67,91.65,10,0,0.945191,0.094519,31.78,0.400000,0.000000
3,2026-05-30-DC-MTL|3|10|1|False,2026-05-30-DC-MTL,o_line,10.55,106.89,96.34,11,0,1.040948,0.094632,40.00,0.454545,0.272727
4,2026-05-30-DC-MTL|3|8|1|False,2026-05-30-DC-MTL,o_line,9.83,108.05,98.22,10,0,0.951760,0.095176,24.26,0.300000,0.350000


## Overlay Highest Possessions

In [7]:
highest_lookup = {path['possession_id'].iloc[0]: path for path in highest_paths}
highest_labeled_paths = {
    f'top {rank + 1}: {row[METRIC]:.3f}': highest_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(highest_possessions.iterrows())
    if row['possession_id'] in highest_lookup
}

if highest_labeled_paths:
    fig = plot_representative_paths(
        highest_labeled_paths,
        title=f'{TEAM_ID.title()} highest {len(highest_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## Overlay Middle Possessions

In [8]:
middle_lookup = {path['possession_id'].iloc[0]: path for path in middle_paths}
middle_labeled_paths = {
    f'middle {rank + 1}: {row[METRIC]:.3f}': middle_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(middle_possessions.iterrows())
    if row['possession_id'] in middle_lookup
}

if middle_labeled_paths:
    fig = plot_representative_paths(
        middle_labeled_paths,
        title=f'{TEAM_ID.title()} middle {len(middle_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## League-Wide Top 5 By Team

Use the same default filter as above: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [9]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_possessions, league_paths = build_scoring_possessions(league_throws)

league_top_possessions, league_top_paths_by_team = select_top_aec_possessions_by_team(
    league_possessions,
    league_paths,
    metric=METRIC,
    n=TOP_N,
)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')
print(f'Teams with qualifying possessions: {league_top_possessions["team_id"].nunique() if not league_top_possessions.empty else 0:,}')

League games loaded: 118
League throws loaded: 64,964
League scoring possessions found: 4,673
Teams with qualifying possessions: 22


In [10]:
league_summary_columns = [
    'team_id', 'team_rank', 'possession_id', 'GameID', 'line_type',
    'start_y', 'end_y', 'field_progress', 'throw_count', 'huck_count',
    'total_aec', METRIC, 'shape_width', 'shape_middle_third_share',
    'shape_sideline_share',
]
league_top_table = league_top_possessions.reindex(columns=league_summary_columns)
print(f'Prepared top {TOP_N} aEC/T possessions for {league_top_table["team_id"].nunique():,} teams.')

Prepared top 5 aEC/T possessions for 22 teams.


## Compare aEC Per Throw vs Total aEC

`aec_per_throw` finds the most efficient possessions per pass. `total_aec` finds possessions with the most accumulated value across the whole point. These can be different, especially when a longer possession has more total value but lower value per throw.

In [11]:
league_metric_comparison = compare_top_aec_metrics_by_team(
    league_possessions,
    league_paths,
    metrics=('aec_per_throw', 'total_aec'),
    n=TOP_N,
)

league_top_by_metric = league_metric_comparison['by_metric']
league_top_by_aec_per_throw, league_top_by_aec_per_throw_paths = league_top_by_metric['aec_per_throw']
league_top_by_total_aec, league_top_by_total_aec_paths = league_top_by_metric['total_aec']
league_metric_overlap = league_metric_comparison['overlap']
league_metric_overlap_table = league_metric_overlap[
    ['team_id', 'overlap_count', 'overlap_share', 'only_aec_per_throw', 'only_total_aec']
]
print('Prepared aEC/T vs total-aEC comparison. Run the side-by-side plot cell below.')

Prepared aEC/T vs total-aEC comparison. Run the side-by-side plot cell below.


In [12]:
comparison_table_columns = [
    'selection_metric', 'team_id', 'team_rank', 'possession_id', 'GameID',
    'throw_count', 'total_aec', 'aec_per_throw', 'selection_value',
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]

league_metric_table = (
    pd.concat(
        [league_top_by_aec_per_throw, league_top_by_total_aec],
        ignore_index=True,
    )
    .sort_values(['team_id', 'selection_metric', 'team_rank'])
    .reset_index(drop=True)
)
league_metric_table = league_metric_table.reindex(columns=comparison_table_columns)
print(f'Prepared {len(league_metric_table):,} comparison rows for optional inspection.')

Prepared 220 comparison rows for optional inspection.


## League-Wide Top 5 Overall

This ranks the full league-wide filtered pool, so these are the top five qualifying possessions overall rather than the top five from each team.

In [13]:
league_top_overall, league_top_overall_paths = select_top_aec_possessions_league(
    league_possessions,
    league_paths,
    metric=METRIC,
    n=TOP_N,
)

metric_label = 'aEC/T' if METRIC == 'aec_per_throw' else METRIC.replace('_', ' ')
league_top_overall_labeled_paths = {
    f"{int(row['league_rank'])}. {str(row['team_id']).title()} {row[METRIC]:.3f}": path
    for (_, row), path in zip(league_top_overall.iterrows(), league_top_overall_paths)
}

plot_representative_paths(
    league_top_overall_labeled_paths,
    title=f"League top {TOP_N} non-huck long-field scoring possessions by {metric_label}",
)

## Team Comparison Browser

In [ ]:
import importlib
import ufa_aec_possessions.plotting
import ufa_aec_possessions.browser

importlib.reload(ufa_aec_possessions.plotting)
importlib.reload(ufa_aec_possessions.browser)

<module 'ufa_aec_possessions.browser' from 'c:\\Users\\htaus\\Desktop\\ufa-aec-possessions\\src\\ufa_aec_possessions\\browser.py'>

In [15]:
team_aec_browser = create_team_aec_comparison_browser(league_metric_comparison)
team_aec_browser